<br>
<a href="https://github.com/aperture-systems-lab">
    <img src="assets/banner_semillero.png" width="955" style="margin: 0px 0px 12px;"/>
</a>
<h1 style="line-height: 1.4;"><font color="#29c4d9"><b>Cómo funcionan las redes neuronales</b></font></h1>
<h2><b>Notebook 4: </b>Tic-tac-toe</h2>

In [ ]:
import random
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim

import utils

utils.set_seeds(42)

----

<br>

## **Parte 0:** AlphaGo

<div style="float: right; width: 46%; min-width: 220px; max-width: 420px; margin: 4px 50px 12px 30px;">
  <img src="assets/alphago.png" width="100%" alt="Lee Sedol jugando contra AlphaGo en 2016"
       style="display: block; transform: rotate(-1.5deg);
              filter: drop-shadow(0px 12px 20px rgba(41, 196, 217, 0.35));"/>
</div>

El Go es un juego de mesa chino de hace más de 2.500 años: un tablero de 19×19, piedras blancas y negras, y unas reglas que se explican en cinco minutos. Lo difícil es jugarlo bien: tiene más posiciones posibles (~10¹⁷⁰) que átomos en el universo (~10⁸⁰), así que ningún computador puede revisar todas las jugadas. Por eso, aunque una máquina le ganó al campeón mundial de ajedrez en 1997, se decía que en el Go faltaban por lo menos diez años más.

En marzo de 2016, AlphaGo de DeepMind (Google) le ganó 4 a 1 a Lee Sedol, uno de los mejores jugadores del mundo (el de la foto, con cara de en qué me metí). No lo logró revisando todas las jugadas, sino con redes neuronales que primero aprendieron de partidas humanas y después mejoraron jugando millones de partidas contra sí mismas.

Hoy no lo vamos a hacer con Go, pero sí con tic-tac-toe (el triqui, el gato, 3 en raya ): una red neuronal que aprende a jugar sin que nadie le enseñe estrategias, solo jugando contra sí misma.

<div style="clear: both;"></div>

----

<br>

## **Parte 1:** Deep Q-Learning

In [ ]:
GAMES = 6_000         
GAMMA = 0.9                    
EPSILON_START = 1.0            
EPSILON_END = 0.1
EPSILON_DECAY = 0.9995        
LEARNING_RATE = 0.001          
LEARNING_RATE_END = 0.00005    
BATCH_SIZE = 64
MEMORY_SIZE = 100_000          

In [ ]:
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(9, 128)
        self.layer_2 = nn.Linear(128, 64)
        self.layer_3 = nn.Linear(64, 9)   
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer_1(x))
        x = self.relu(self.layer_2(x))
        return self.layer_3(x)

----

<br>

## **Parte 2:** Entrenamiento

In [ ]:
@torch.no_grad()
def choose_action(model, board, epsilon=0.0):
    cells = utils.free_cells(board)
    if random.random() < epsilon:
        return random.choice(cells)                            
    q = model(torch.tensor(board, dtype=torch.float32))
    return max(cells, key=lambda cell: q[cell])               


def learn(model, optimizer, memory):
    boards, cells, rewards, next_boards, dones = zip(*random.sample(memory, BATCH_SIZE))
    boards = torch.tensor(boards, dtype=torch.float32)
    next_boards = torch.tensor(next_boards, dtype=torch.float32)

    q = model(boards)[torch.arange(BATCH_SIZE), torch.tensor(cells)]     

    with torch.no_grad():                                            
        q_rival = model(next_boards).masked_fill(next_boards != 0, float("-inf")).max(dim=1).values
        target = torch.where(torch.tensor(dones), torch.tensor(rewards), -GAMMA * q_rival)

    loss = nn.functional.mse_loss(q, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
model = QNetwork()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0,
                                        end_factor=LEARNING_RATE_END / LEARNING_RATE, total_iters=GAMES)
memory = deque(maxlen=MEMORY_SIZE)
epsilon = EPSILON_START

for game in range(1, GAMES + 1):
    board, done = utils.EMPTY, False

    while not done:
        cell = choose_action(model, board, epsilon)                          
        next_board, reward, done = utils.step(board, cell)                  
        memory.extend(utils.symmetries((board, cell, reward, next_board, done))) 
        if len(memory) >= BATCH_SIZE:
            learn(model, optimizer, memory)                                     
        board = next_board

    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    scheduler.step()

    if game % 1000 == 0:
        print(f"Partida {game:5d} — ε = {epsilon:.2f}")

----

<br>

## **Parte 3:** A jugar

In [ ]:
utils.play_against(model)

-----

<br>

### <font color="#29c4d9">**Notebook 4 listo.**</font>

<br>

---

<div style="margin-top: 50px;"><center><a href="https://github.com/aperture-systems-lab"><img src="assets/banner_logo.png" width="955"/></a></center></div>